# Análise de Keywords de Controle de Acesso

Este notebook carrega `script_pncp/config/keywords_acesso.txt`, varre o banco de resultados gerado pelo BART e produz: contagens por termo, uma tabela com as top correspondências e amostras para revisão.

Use as células abaixo para inspecionar, ajustar o comportamento (uso de word-boundaries) e exportar resultados para CSV/JSON.

In [1]:
# 1. Configurações de caminhos
from pathlib import Path
import sqlite3
import pandas as pd
import json

KW_FILE = Path('/Users/jose-cleiton/dev/script_pncp/config/keywords_acesso.txt')
DB_PATH = Path('/Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/pncp_filtrado_bart_1000.db')
OUT_CSV = Path('/Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/keywords_acesso_matches_sample_from_notebook.csv')
OUT_JSON = Path('/Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/keywords_acesso_counts_from_notebook.json')

print('KW_FILE:', KW_FILE)
print('DB_PATH:', DB_PATH)

KW_FILE: /Users/jose-cleiton/dev/script_pncp/config/keywords_acesso.txt
DB_PATH: /Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/pncp_filtrado_bart_1000.db


In [2]:
# 2. Carregar keywords
def load_keywords(path: Path):
    text = path.read_text(encoding='utf-8')
    kws = [l.strip() for l in text.splitlines() if l.strip()]
    return kws

keywords = load_keywords(KW_FILE)
print(f'Loaded {len(keywords)} keywords')
# show first 20
keywords[:20]

Loaded 53 keywords


['software de controle de acesso',
 'software de segurança',
 'software de monitoramento',
 'software para controle de portaria',
 'software para registro de frequência',
 'software de gestão',
 'relógio de ponto',
 'cartão de ponto',
 'controle de ponto',
 'reconhecimento facial',
 'registrador eletrônico de ponto biométrico',
 'gerenciamento eletrônico de frequência',
 'controle de frequência',
 'registrador de ponto facial',
 'sistema com aplicativo facial',
 'controle de frequência facial',
 'controle de faces',
 'equipamento com biometria de face',
 'identificador facial',
 'bastão de ronda']

In [5]:
# 3. Função de matching (defina antes da varredura)
import re

def match_keywords(text: str, keywords: list, use_word_boundaries_for_short=True):
    if not text:
        return []
    t = text.lower()
    matches = []
    for kw in keywords:
        kl = kw.lower()
        if use_word_boundaries_for_short and len(kl) <= 4:
            if re.search(r'\b' + re.escape(kl) + r'\b', t):
                matches.append(kw)
        else:
            if kl in t:
                matches.append(kw)
    return matches

# quick self-check
print('matches example:', match_keywords('Instalação de HID leitor de cartões', ['hid','leitor facial']))

matches example: ['hid']


In [6]:
# 4. Varredura no DB: contagens por termo e amostras
from collections import Counter
counts = Counter()
samples = []
max_samples = 20
use_word_boundaries_for_short = True  # ajuste aqui se quiser testar

if not DB_PATH.exists():
    raise FileNotFoundError(f'DB não encontrado: {DB_PATH}')

with sqlite3.connect(str(DB_PATH)) as conn:
    cur = conn.cursor()
    cur.execute('SELECT pncp_id, numero_controle, objeto_compra, prob_relevante, categoria, motivo FROM contratacoes_filtradas')
    for row in cur:
        pncp_id, numero_controle, objeto_compra, prob_relevante, categoria, motivo = row
        texto = objeto_compra or ''
        matched = match_keywords(texto, keywords, use_word_boundaries_for_short)
        for m in matched:
            counts[m] += 1
        if matched and len(samples) < max_samples:
            samples.append({
                'pncp_id': pncp_id,
                'numero_controle': numero_controle,
                'objeto_compra': objeto_compra,
                'prob_relevante': prob_relevante,
                'categoria': categoria,
                'motivo': motivo,
                'matched_keywords': matched
            })

# convert counts to DataFrame for nicer display
counts_df = pd.DataFrame(sorted(list(counts.items()), key=lambda x: x[1], reverse=True), columns=['keyword','count'])
print('Total matches (sum counts):', sum(counts.values()))
print('Unique keywords with matches:', (counts_df['count']>0).sum())
counts_df.head(50)

Total matches (sum counts): 2
Unique keywords with matches: 1


,keyword,count
0,controle de acesso,2


In [7]:
# 5. Mostrar amostras e salvar resultados
samples_df = pd.DataFrame(samples) if samples else pd.DataFrame(columns=['pncp_id','numero_controle','objeto_compra','prob_relevante','categoria','motivo','matched_keywords'])
display(samples_df.head(20))
# salvar outputs
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(dict(counts), ensure_ascii=False, indent=2), encoding='utf-8')
samples_df.to_csv(OUT_CSV, index=False)
print('Saved JSON:', OUT_JSON)
print('Saved CSV samples:', OUT_CSV)

,pncp_id,numero_controle,objeto_compra,prob_relevante,categoria,motivo,matched_keywords
0,29844172000123-1-000011/2026,29844172000123-1-000011/2026,Prestação de serviços de locação de equipament...,0.999991,1.0,Termos positivos fortes encontrados: controle ...,[controle de acesso]
1,18413161000172-1-000011/2026,18413161000172-1-000011/2026,Registro de preços para futura contratação de ...,0.999990,1.0,Termos positivos fortes encontrados: controle ...,[controle de acesso]


Saved JSON: /Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/keywords_acesso_counts_from_notebook.json
Saved CSV samples: /Users/jose-cleiton/dev/script_pncp/data/resultado/2026-03-25/keywords_acesso_matches_sample_from_notebook.csv


## Próximos passos e notas rápidas

- Se `hid` aparecer muito, experimente definir `use_word_boundaries_for_short = False` ou remova `hid` do arquivo de keywords.
- Você pode integrar `match_keywords` no runner de classificação para gravar `matched_keywords` e `kw_flags` na tabela `contratacoes_filtradas`. Recomendo fazer isso inicialmente sem alterar `categoria` (modo seguro).
- Para usar essas flags no treino, adicione uma coluna `has_kw` no CSV de treino ou prefira o prefixo no texto (`[KW: ...]`) para sinalizar ao modelo.

Células:
1: Descrição, 2: caminhos, 3: carregar keywords, 4: função de matching, 5: varredura no DB e contagens, 6: amostras e salvamento.